# Week 3 Lab. Zero-shot versus Few-shot A/B and Chain-of-Thought Uplift

**Instructor solution. Fully worked, including stretch goals.**

You already trust unit tests, precision and recall, and A/B comparisons from
regular engineering. This lab points those same instincts at model behavior. You
will build the machinery that turns "the prompt feels better" into a number you
can defend.

**Learning objectives.** By the end you can:

1. Build zero-shot and few-shot prompts with an explicit output contract.
2. Parse messy model output and score it with precision, recall, F1, and a
   confusion matrix.
3. Show that two prompts with the same accuracy can behave very differently per
   class, and read the confusion matrix to see why.
4. Measure chain-of-thought uplift, stabilize it with self-consistency, and
   weigh the accuracy gain against token cost.
5. Ship a private reason-then-answer contract and guard against rationale
   leakage.

**How this lab runs.** Calling a model live in a classroom is slow, costs money,
and gives a different answer every run, which makes grading impossible. So the
model responses are captured once and shipped as fixtures under `fixtures/`.
Your job is the engineering around the model: the prompts, the parsing, the
scoring, and the analysis. Your instructor will show the live Claude call that
produced these fixtures during the demo. The fixtures are representative captured
runs. Live numbers will move, but the shape of the result holds.

**Time budget.** About 100 minutes. Part 1 near 45 minutes, Part 2 near 50
minutes, stretch goals if you finish early.

**Working the lab.** Each task has a stub that raises `NotImplementedError` and a
`check()` call right below it. Run a cell to see it fail, implement the function,
re-run to turn it green. Nothing here hard-crashes, so run freely. Call
`summary()` at the end for your score. If you get stuck, `HINTS.md` has three
escalating levels per task.

In [1]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

DATA = Path("data")
FIX = Path("fixtures")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

REVIEWS = load_json(DATA / "reviews.json")
COT = load_json(DATA / "cot_tasks.json")

LABELS = REVIEWS["labels"]
EVAL_ITEMS = REVIEWS["eval"]
GOLD_CLS = {e["id"]: e["gold"] for e in EVAL_ITEMS}
COT_ITEMS = COT["mwp"] + COT["pol"]
GOLD_COT = COT["gold"]

def read_fixture(name):
    return (FIX / name).read_text(encoding="utf-8")

# Grading inputs, independent of any student function.
REF_ZERO_PRED = {
    "E01": "bug_report", "E02": "feature_request", "E03": "praise",
    "E04": "question", "E05": "other", "E06": "feature_request",
    "E07": "praise", "E08": "question", "E09": "feature_request",
    "E10": "praise",
}
REF_NOCOT_PRED = {"M01": 113, "M02": 1600, "M03": 23, "P01": "deny",
                  "P02": "deny", "P03": "approve"}
REF_CONSENSUS_RUNS = [
    {"M01": 113, "M02": 1.6, "M03": 28, "P01": "reimburse_meal_only", "P02": "approve", "P03": "approve"},
    {"M01": 113, "M02": 1.6, "M03": 28, "P01": "reimburse_meal_only", "P02": "deny", "P03": "approve"},
    {"M01": 113, "M02": 1.6, "M03": 33, "P01": "reimburse_meal_only", "P02": "deny", "P03": "approve"},
    {"M01": 113, "M02": 1.6, "M03": 28, "P01": "reimburse_meal_only", "P02": "deny", "P03": "approve"},
    {"M01": 113, "M02": 1.6, "M03": 28, "P01": "reimburse_meal_only", "P02": "approve", "P03": "approve"},
]
REF_PRIV_CLEAN = [{"id": "M01", "final_answer": 113}, {"id": "P02", "final_answer": "deny"}]
REF_PRIV_LEAKY = [
    {"id": "M01", "final_answer": 113},
    {"id": "M02", "final_answer": 1.6, "rationale": "1600 m to km"},
    {"id": "P02", "final_answer": "deny because it is after the cutoff"},
]

# Curated few-shot examples (provided, so you can focus on the eval machinery).
POOL = {s["id"]: s for s in REVIEWS["example_pool"]}
FEWSHOT_EASY_TO_HARD = [POOL[i] for i in ["S01", "S02", "S06", "S09", "S10", "S12"]]
FEWSHOT_INTERLEAVE = [POOL[i] for i in ["S01", "S06", "S10", "S02", "S09", "S12"]]

_RESULTS = {}
CORE_TASKS = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10"]

def _approx(a, b, tol=1e-3):
    return abs(float(a) - float(b)) <= tol

def _verify(task, got):
    if task == "T1":
        need = ["<INSTRUCTION>", "<OUTPUT_CONTRACT>", "<INPUTS>", "records", "stats"]
        miss = [n for n in need if n not in got]
        miss += [l for l in LABELS if l not in got]
        miss += [e["id"] for e in EVAL_ITEMS if e["id"] not in got]
        return (not miss, "missing " + ", ".join(map(str, miss)) if miss else "")
    if task == "T2":
        need = ["<EXAMPLES>", "<QUERY>", "<OUTPUT_CONTRACT>"]
        miss = [n for n in need if n not in got]
        miss += [ex["text"] for ex in FEWSHOT_EASY_TO_HARD if ex["text"] not in got]
        return (not miss, "missing examples or sections" if miss else "")
    if task == "T3":
        ids = [r["id"] for r in got]
        by = {r["id"]: r["label"] for r in got}
        ok = len(got) == 10 and by.get("E05") == "other" and ids[0] == "E01"
        return (ok, f"parsed {len(got)} records, E05={by.get('E05')}")
    if task == "T4":
        ok = (_approx(got["macro"]["f1"], 0.5933) and _approx(got["accuracy"], 0.7)
              and got["confusion"]["bug_report"]["question"] == 1)
        return (ok, f"macro_f1={got['macro']['f1']:.4f} acc={got['accuracy']:.3f}")
    if task == "T5":
        yes, no = got["yes"], got["no"]
        ok = ("rationale" in yes and "rationale" not in no
               and all(it["id"] in yes and it["id"] in no for it in COT_ITEMS))
        return (ok, "rationale gating or item coverage wrong")
    if task == "T6":
        ok = (_approx(got["a"], 1.6) and got["b"] == "deny" and _approx(got["c"], 28))
        return (ok, f"got {got}")
    if task == "T7":
        ok = got["correct"] == 3 and _approx(got["accuracy"], 0.5)
        return (ok, f"acc={got['accuracy']:.3f} correct={got['correct']}")
    if task == "T8":
        ok = got.get("P02") == "deny" and str(got.get("M03")) == "28"
        return (ok, f"P02={got.get('P02')} M03={got.get('M03')}")
    if task == "T9":
        ok = got["no_cot"] == 75 and got["cot_single"] == 153
        return (ok, f"no_cot={got['no_cot']} cot_single={got['cot_single']}")
    if task == "T10":
        ok = got["clean"]["clean"] is True and set(got["leaky"]["offenders"]) == {"M02", "P02"}
        return (ok, f"clean={got['clean']} leaky={got['leaky']}")
    # stretch tasks
    if task == "S1":
        ok = _approx(got["macro_f1"], 0.5933) and _approx(got["weighted_f1"], 0.6767, tol=0.01)
        return (ok, f"macro={got['macro_f1']:.4f} weighted={got['weighted_f1']:.4f}")
    if task == "S2":
        ok = got["pair"] == ("bug_report", "question") or got["pair"] == ("bug_report", "other")
        return (ok, f"most-confused pair {got['pair']}")
    if task == "S3":
        ok = got["repaired"] >= 1 and got["records"] == 10
        return (ok, f"repaired={got['repaired']} records={got['records']}")
    return (False, "unknown task")

def check(task, thunk):
    try:
        got = thunk()
    except NotImplementedError:
        _RESULTS[task] = "todo"
        print(f"[ ] {task}: not implemented yet")
        return
    except Exception as e:
        _RESULTS[task] = "error"
        print(f"[!] {task}: raised {type(e).__name__}: {e}")
        return
    ok, detail = _verify(task, got)
    _RESULTS[task] = "pass" if ok else "fail"
    tag = "PASS" if ok else "FAIL"
    print(f"[{'x' if ok else ' '}] {task}: {tag}" + (f"  ({detail})" if detail and not ok else ""))

def summary():
    passed = sum(1 for t in CORE_TASKS if _RESULTS.get(t) == "pass")
    print(f"\nCORE: {passed}/{len(CORE_TASKS)} passing")
    stretch = [t for t in ("S1", "S2", "S3") if _RESULTS.get(t) == "pass"]
    if stretch:
        print(f"STRETCH passing: {', '.join(stretch)}")
    if passed < len(CORE_TASKS):
        todo = [t for t in CORE_TASKS if _RESULTS.get(t) != "pass"]
        print("Still open: " + ", ".join(todo))
    else:
        print("All core tasks green.")

print("Harness ready. Data and fixtures loaded.")

Harness ready. Data and fixtures loaded.


---
## Part 1. Zero-shot versus few-shot, measured

You have a small labelled set of app-store style reviews for a fictional product
from Cordwell Home and Hardware. Five labels: `bug_report`, `feature_request`,
`praise`, `question`, `other`. You will classify ten held-out reviews three
ways, zero-shot and two few-shot orderings, then score them.

### Provided. The few-shot examples and two orderings

Curating examples is its own skill, so it is done for you here. Six examples were
chosen from a pool of twelve: two typical, two edge, and two adversarial. The
adversarial pair matters most. `S10` is a bug phrased as a question, and `S12` is
mixed sentiment that resolves to `other`. Those two teach the boundaries the
model gets wrong without help.

Two orderings are provided, and the fixtures correspond to them:

- Easy to hard: typical, typical, edge, edge, adversarial, adversarial.
- Interleaved: typical, edge, adversarial, repeated.

Watch what each ordering does to the per-class scores later. Same six examples,
different order, different behavior.

### Task 1. Build the zero-shot prompt

Write `build_zero_shot_prompt(labels, items)`. It returns a single prompt
string that a model would receive.

Contract:
- Names every allowed label from `labels`.
- Contains an `<INSTRUCTION>`, an `<OUTPUT_CONTRACT>`, and an `<INPUTS>` section.
- The output contract describes a JSON object with a `records` array and a
  `stats` object holding a `count`.
- Lists every eval item as `id: text`.

In [2]:
def build_zero_shot_prompt(labels, items):
    label_line = ", ".join(labels)
    lines = [
        "<INSTRUCTION>",
        f"Classify each review into exactly one of: {label_line}.",
        "Return STRICT JSON only per OUTPUT_CONTRACT. No prose outside the JSON.",
        "</INSTRUCTION>",
        "",
        "<OUTPUT_CONTRACT>",
        '{"records":[{"id":"<E##>","label":"<one_label>","reasons":["<why>"]}],"stats":{"count":"<int>"}}',
        "</OUTPUT_CONTRACT>",
        "",
        "<INPUTS>",
    ]
    for it in items:
        lines.append(f'{it["id"]}: {it["text"]}')
    lines.append("</INPUTS>")
    return "\n".join(lines)

check("T1", lambda: build_zero_shot_prompt(LABELS, EVAL_ITEMS))

[x] T1: PASS


### Task 2. Build the few-shot prompt

Write `build_few_shot_prompt(labels, examples, items)`. Same idea as the
zero-shot prompt, but it also teaches the model with labelled examples before
the query.

Contract:
- Contains an `<EXAMPLES>` section listing each example with its label and text.
- Contains a `<QUERY>` section listing the eval items as `id: text`.
- Keeps the same `<OUTPUT_CONTRACT>`.

In [3]:
def build_few_shot_prompt(labels, examples, items):
    label_line = ", ".join(labels)
    lines = [
        "<INSTRUCTION>",
        f"Classify each review into exactly one of: {label_line}.",
        "Study the EXAMPLES to learn the label boundaries, then classify the QUERY.",
        "Return STRICT JSON only per OUTPUT_CONTRACT. No prose outside the JSON.",
        "</INSTRUCTION>",
        "",
        "<OUTPUT_CONTRACT>",
        '{"records":[{"id":"<E##>","label":"<one_label>","reasons":["<why>"]}],"stats":{"count":"<int>"}}',
        "</OUTPUT_CONTRACT>",
        "",
        "<EXAMPLES>",
    ]
    for ex in examples:
        lines.append(f'[{ex["label"]}] {ex["text"]}')
    lines.append("</EXAMPLES>")
    lines.append("")
    lines.append("<QUERY>")
    for it in items:
        lines.append(f'{it["id"]}: {it["text"]}')
    lines.append("</QUERY>")
    return "\n".join(lines)

check("T2", lambda: build_few_shot_prompt(LABELS, FEWSHOT_EASY_TO_HARD, EVAL_ITEMS))

[x] T2: PASS


### Task 3. Parse records from a raw response

Real model output is a string, sometimes wrapped in a code fence, sometimes
with a stray sentence before or after. Write `parse_records(raw)` that returns
the `records` list from whatever the model sent back.

Contract:
- Accepts a raw string.
- Tolerates a surrounding Markdown code fence and leading or trailing prose.
- Returns the `records` list (a list of dicts).
- Raises `ValueError` when no JSON object can be found.

In [4]:
def parse_records(raw):
    text = raw.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.DOTALL)
    if fence:
        text = fence.group(1).strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("no JSON object found in response")
    obj = json.loads(text[start:end + 1])
    records = obj.get("records")
    if not isinstance(records, list):
        raise ValueError("parsed object has no 'records' list")
    return records

check("T3", lambda: parse_records(read_fixture("cls_zero_shot.txt")))

[x] T3: PASS


### Task 4. Score the classification

Write `score_classification(gold, pred, labels)`. This is the eval harness: it
is unit testing for model behavior. Return per-label precision, recall, and F1,
the macro averages, overall accuracy, and a confusion matrix keyed
`confusion[true][pred]`.

Contract:
- `gold` and `pred` are dicts mapping id to a label string.
- When an id is missing from `pred`, treat its prediction as `"other"`.
- Return a dict with keys `per_label`, `macro`, `accuracy`, and `confusion`.
- `per_label[label]` has `precision`, `recall`, `f1`, and `support`.
- `macro` averages each metric evenly across all labels.

In [5]:
def score_classification(gold, pred, labels):
    cm = defaultdict(Counter)
    for _id, g in gold.items():
        p = pred.get(_id, "other")
        cm[g][p] += 1
    per_label = {}
    for l in labels:
        tp = cm[l][l]
        fp = sum(cm[t][l] for t in labels if t != l)
        fn = sum(cm[l][p] for p in labels if p != l)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        per_label[l] = {"precision": prec, "recall": rec, "f1": f1,
                        "support": sum(cm[l].values())}
    n = len(labels)
    macro = {m: sum(v[m] for v in per_label.values()) / n
             for m in ("precision", "recall", "f1")}
    accuracy = sum(cm[l][l] for l in labels) / sum(sum(cm[t].values()) for t in labels)
    confusion = {t: {p: cm[t][p] for p in labels} for t in labels}
    return {"per_label": per_label, "macro": macro, "accuracy": accuracy,
            "confusion": confusion}

check("T4", lambda: score_classification(GOLD_CLS, REF_ZERO_PRED, LABELS))

[x] T4: PASS


### Run the A and B comparison

With Task 3 and Task 4 done, this cell scores all three response fixtures. Read
the confusion matrices, not just the headline accuracy.

In [6]:
def run_classification(fixture_name):
    recs = parse_records(read_fixture(fixture_name))
    pred = {r["id"]: r["label"] for r in recs}
    return score_classification(GOLD_CLS, pred, LABELS)

def print_report(name, res):
    print(f"== {name} ==")
    print(f"  accuracy = {res['accuracy']:.3f}   macro F1 = {res['macro']['f1']:.3f}")
    for l in LABELS:
        m = res["per_label"][l]
        print(f"    {l:16s} P={m['precision']:.2f} R={m['recall']:.2f} "
              f"F1={m['f1']:.2f} n={m['support']}")
    head = "        " + " ".join(f"{l[:5]:>5}" for l in LABELS)
    print("  confusion (true rows, predicted cols):")
    print(head)
    for t in LABELS:
        row = " ".join(f"{res['confusion'][t][p]:>5d}" for p in LABELS)
        print(f"    {t[:5]:>5} {row}")
    print()

try:
    for nm, fx in [("zero-shot", "cls_zero_shot.txt"),
                   ("few-shot easy-to-hard", "cls_few_easy_to_hard.txt"),
                   ("few-shot interleave", "cls_few_interleave.txt")]:
        print_report(nm, run_classification(fx))
except NotImplementedError:
    print("Implement Task 3 and Task 4, then re-run this cell.")

== zero-shot ==
  accuracy = 0.700   macro F1 = 0.593
    bug_report       P=1.00 R=0.33 F1=0.50 n=3
    feature_request  P=1.00 R=1.00 F1=1.00 n=3
    praise           P=0.67 R=1.00 F1=0.80 n=2
    question         P=0.50 R=1.00 F1=0.67 n=1
    other            P=0.00 R=0.00 F1=0.00 n=1
  confusion (true rows, predicted cols):
        bug_r featu prais quest other
    bug_r     1     0     0     1     1
    featu     0     3     0     0     0
    prais     0     0     2     0     0
    quest     0     0     0     1     0
    other     0     0     1     0     0

== few-shot easy-to-hard ==
  accuracy = 0.900   macro F1 = 0.893
    bug_report       P=1.00 R=0.67 F1=0.80 n=3
    feature_request  P=1.00 R=1.00 F1=1.00 n=3
    praise           P=1.00 R=1.00 F1=1.00 n=2
    question         P=1.00 R=1.00 F1=1.00 n=1
    other            P=0.50 R=1.00 F1=0.67 n=1
  confusion (true rows, predicted cols):
        bug_r featu prais quest other
    bug_r     2     0     0     0     1
    featu  

### Read the result before you move on

Two things worth saying out loud:

- Few-shot lifts macro F1 well above zero-shot. The examples pin down the
  boundaries the zero-shot prompt was guessing at.
- The two few-shot orderings reach the same accuracy yet different macro F1.
  Accuracy is one number over all items. Macro F1 averages the per-class F1, so
  it exposes a class that quietly fell apart. When one ordering fixes one
  boundary but breaks another, the confusion matrix is where you see it.

That is the whole point of a harness. A single number can hide a regression that
per-class metrics surface immediately.

---
## Part 2. Chain-of-thought uplift, then pay for it

Now a reasoning task: three math word problems and three expense-policy checks,
each with a known answer. You will compare a direct-answer prompt against a
chain-of-thought prompt, stabilize the reasoning with self-consistency, and then
count the tokens it costs.

### Task 5. Build the CoT prompt

Write `build_cot_prompt(items, use_cot)`. One function, two modes.

Contract:
- When `use_cot` is False, the instruction asks only for the final answer and
  the contract has records of `id` and `final_answer`.
- When `use_cot` is True, the instruction asks the model to reason step by step
  and the contract adds a `rationale` field.
- Both modes list every item as `id: text` inside `<INPUTS>`.

In [7]:
def build_cot_prompt(items, use_cot):
    if use_cot:
        instruction = ("Think step by step. Show a brief rationale, verify the "
                       "arithmetic or policy condition, then give the final answer.")
        contract = ('{"records":[{"id":"<ID>","final_answer":"<value_or_label>",'
                    '"rationale":"<2 to 6 short steps>"}],"stats":{"count":"<int>"}}')
    else:
        instruction = "Solve each item and give only the final answer."
        contract = ('{"records":[{"id":"<ID>","final_answer":"<value_or_label>"}],'
                    '"stats":{"count":"<int>"}}')
    lines = ["<INSTRUCTION>", instruction,
             "Return STRICT JSON only per OUTPUT_CONTRACT.", "</INSTRUCTION>", "",
             "<OUTPUT_CONTRACT>", contract, "</OUTPUT_CONTRACT>", "", "<INPUTS>"]
    for it in items:
        lines.append(f'{it["id"]}: {it["text"]}')
    lines.append("</INPUTS>")
    return "\n".join(lines)

check("T5", lambda: {"yes": build_cot_prompt(COT_ITEMS, True),
                     "no": build_cot_prompt(COT_ITEMS, False)})

[x] T5: PASS


### Task 6. Normalize an answer

Model answers arrive as numbers or strings, sometimes as `"1.6"` and sometimes
as `1.6`. Write `normalize_answer(v)` so equal answers compare equal.

Contract:
- Numbers return as `float`.
- Numeric strings return as `float`.
- Other strings return lower-cased and stripped.
- A `bool` returns unchanged.

In [8]:
def normalize_answer(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        s = v.strip().lower()
        try:
            return float(s)
        except ValueError:
            return s
    return v

check("T6", lambda: {"a": normalize_answer("1.6"),
                     "b": normalize_answer(" Deny "),
                     "c": normalize_answer(28)})

[x] T6: PASS


### Task 7. Score the CoT answers

Write `score_cot(gold, pred)`. Compare predictions to gold using
`normalize_answer` so `"1.6"` matches `1.6`.

Contract:
- `gold` and `pred` are dicts mapping id to an answer.
- Return a dict with `accuracy`, `correct`, `total`, and `rows`.
- `rows` is a list of dicts with `id`, `pred`, `gold`, and `correct`.

In [9]:
def score_cot(gold, pred):
    total = len(gold)
    correct = 0
    rows = []
    for k, g in gold.items():
        y = pred.get(k)
        ok = normalize_answer(y) == normalize_answer(g)
        rows.append({"id": k, "pred": y, "gold": g, "correct": ok})
        if ok:
            correct += 1
    return {"accuracy": correct / total, "correct": correct, "total": total, "rows": rows}

check("T7", lambda: score_cot(GOLD_COT, REF_NOCOT_PRED))

[x] T7: PASS


### Task 8. Self-consistency by majority vote

Sampling the same CoT prompt several times and taking the most common final
answer is called self-consistency. Write `consensus(runs)`.

Contract:
- `runs` is a list of dicts, each mapping id to a final answer.
- For each id, return the most common answer across runs.
- Compare answers as their stripped string form so `28` and `"28"` count
  together.
- Return a dict mapping id to the winning answer (as a string).

In [10]:
def consensus(runs):
    votes = defaultdict(Counter)
    for run in runs:
        for _id, ans in run.items():
            votes[_id][str(ans).strip()] += 1
    return {_id: ctr.most_common(1)[0][0] for _id, ctr in votes.items()}

check("T8", lambda: consensus(REF_CONSENSUS_RUNS))

[x] T8: PASS


### Task 9. Estimate token cost

Write `estimate_tokens(text)`, a coarse proxy of about one token per four
characters. It is only for relative comparison, not for billing. Real token
counts come from the model response usage field.

Contract:
- Return an int, at least 1.
- Use about `len(text) / 4`, rounded.

In [11]:
def estimate_tokens(text):
    return max(1, round(len(text) / 4))

check("T9", lambda: {"no_cot": estimate_tokens(read_fixture("cot_no.txt")),
                     "cot_single": estimate_tokens(read_fixture("cot_yes_run1.txt"))})

[x] T9: PASS


### Task 10. Guard against rationale leakage

In production you often want the model to reason but expose only a clean final
answer. Write `check_no_leakage(records)` to catch responses that leak
reasoning.

Contract:
- A record is clean when its only keys are `id` and `final_answer`.
- A record also leaks when its `final_answer` string contains a reasoning
  marker: `step`, `because`, `therefore`, `first,`, or `reason`.
- Return a dict `{"clean": bool, "offenders": [ids...]}`.

In [12]:
def check_no_leakage(records):
    allowed = {"id", "final_answer"}
    markers = ("step", "because", "therefore", "first,", "reason")
    offenders = []
    for r in records:
        extra = set(r.keys()) - allowed
        fa = str(r.get("final_answer", "")).lower()
        if extra or any(m in fa for m in markers):
            offenders.append(r.get("id"))
    return {"clean": not offenders, "offenders": offenders}

check("T10", lambda: {"clean": check_no_leakage(REF_PRIV_CLEAN),
                      "leaky": check_no_leakage(REF_PRIV_LEAKY)})

[x] T10: PASS


### Run the uplift comparison

With Tasks 3, 6, 7, and 8 done, this scores no-CoT, a single CoT run, the K=5
majority vote, and the private variant, and shows exactly where the vote
overturned a single noisy run.

In [13]:
def cot_pred(fixture_name):
    recs = parse_records(read_fixture(fixture_name))
    return {r["id"]: r["final_answer"] for r in recs}

try:
    no = score_cot(GOLD_COT, cot_pred("cot_no.txt"))
    single = score_cot(GOLD_COT, cot_pred("cot_yes_run1.txt"))
    runs = [cot_pred(f"cot_yes_run{i}.txt") for i in range(1, 6)]
    cons = consensus(runs)
    cons_scored = score_cot(GOLD_COT, cons)
    priv = score_cot(GOLD_COT, cot_pred("cot_private.txt"))

    print(f"no-CoT       accuracy = {no['correct']}/{no['total']} = {no['accuracy']:.3f}")
    print(f"CoT single   accuracy = {single['correct']}/{single['total']} = {single['accuracy']:.3f}")
    print(f"CoT K=5 vote accuracy = {cons_scored['correct']}/{cons_scored['total']} = {cons_scored['accuracy']:.3f}")
    print(f"private      accuracy = {priv['correct']}/{priv['total']} = {priv['accuracy']:.3f}")

    print("\nWhere consensus changed the single-run answer:")
    single_map = cot_pred("cot_yes_run1.txt")
    for _id in sorted(cons):
        votes = Counter(str(r[_id]).strip() for r in runs)
        changed = str(single_map[_id]).strip() != cons[_id]
        flag = "  <- changed by vote" if changed else ""
        print(f"  {_id}: votes={dict(votes)} -> {cons[_id]}{flag}")
except NotImplementedError:
    print("Implement Tasks 3, 6, 7, and 8, then re-run this cell.")

no-CoT       accuracy = 3/6 = 0.500
CoT single   accuracy = 5/6 = 0.833
CoT K=5 vote accuracy = 6/6 = 1.000
private      accuracy = 5/6 = 0.833

Where consensus changed the single-run answer:
  M01: votes={'113': 5} -> 113
  M02: votes={'1.6': 5} -> 1.6
  M03: votes={'28': 4, '33': 1} -> 28
  P01: votes={'reimburse_meal_only': 5} -> reimburse_meal_only
  P02: votes={'approve': 2, 'deny': 3} -> deny  <- changed by vote
  P03: votes={'approve': 5} -> approve


### Count the cost

Reasoning is not free. The rationale is extra output tokens, and self-consistency
multiplies that by the number of samples. This cell shows the tradeoff. The
estimate is a coarse proxy. In production you read the real counts from the model
response usage field.

In [14]:
try:
    p_no = build_cot_prompt(COT_ITEMS, False)
    p_yes = build_cot_prompt(COT_ITEMS, True)
    resp_no = estimate_tokens(read_fixture("cot_no.txt"))
    resp_single = estimate_tokens(read_fixture("cot_yes_run1.txt"))
    resp_k5 = sum(estimate_tokens(read_fixture(f"cot_yes_run{i}.txt")) for i in range(1, 6))
    resp_priv = estimate_tokens(read_fixture("cot_private.txt"))

    print("prompt tokens (proxy):")
    print(f"  no-CoT prompt   ~ {estimate_tokens(p_no)}")
    print(f"  CoT prompt      ~ {estimate_tokens(p_yes)}")
    print("response tokens (proxy):")
    print(f"  no-CoT          ~ {resp_no}")
    print(f"  CoT single      ~ {resp_single}   ({resp_single/resp_no:.1f}x no-CoT)")
    print(f"  CoT K=5 vote    ~ {resp_k5}   ({resp_k5/resp_no:.1f}x no-CoT)")
    print(f"  private answer  ~ {resp_priv}   ({resp_priv/resp_no:.1f}x no-CoT)")
except NotImplementedError:
    print("Implement Tasks 5 and 9, then re-run this cell.")

prompt tokens (proxy):
  no-CoT prompt   ~ 311
  CoT prompt      ~ 337
response tokens (proxy):
  no-CoT          ~ 75
  CoT single      ~ 153   (2.0x no-CoT)
  CoT K=5 vote    ~ 694   (9.3x no-CoT)
  private answer  ~ 79   (1.1x no-CoT)


### Guard the contract

The private variant keeps chain-of-thought accuracy while returning only the
final answer, so users never see the reasoning. The guard catches any record
that leaks a rationale, whether as an extra field or smuggled into the answer
string.

In [15]:
try:
    priv_recs = parse_records(read_fixture("cot_private.txt"))
    print("private fixture leakage check:", check_no_leakage(priv_recs))
    leaky_demo = [{"id": "M02", "final_answer": 1.6, "rationale": "1600 m to km"},
                  {"id": "P02", "final_answer": "deny because after cutoff"}]
    print("leaky demo leakage check: ", check_no_leakage(leaky_demo))
except NotImplementedError:
    print("Implement Tasks 3 and 10, then re-run this cell.")

private fixture leakage check: {'clean': True, 'offenders': []}
leaky demo leakage check:  {'clean': False, 'offenders': ['M02', 'P02']}


## Stretch goals

Optional. Solutions are graded separately from the core ten and do not block a
green run.

### Stretch 1. Support-weighted F1

Macro F1 weights every label equally, even a label with one example. Write
`weighted_f1(res)` that averages each label F1 by its support, and return both
so you can compare. When does each average tell a more useful story?

In [16]:
def weighted_f1(res):
    total = sum(m["support"] for m in res["per_label"].values())
    wf = sum(m["f1"] * m["support"] for m in res["per_label"].values()) / total
    return {"macro_f1": res["macro"]["f1"], "weighted_f1": wf}

check("S1", lambda: weighted_f1(score_classification(GOLD_CLS, REF_ZERO_PRED, LABELS)))

[x] S1: PASS


### Stretch 2. Confusion-driven example selection

Find the most-confused off-diagonal cell in the zero-shot confusion matrix.
That pair tells you which boundary a targeted few-shot example should defend.
Write `most_confused_pair(res)` returning the `(true, predicted)` pair with the
largest off-diagonal count.

In [17]:
def most_confused_pair(res):
    best, best_pair = 0, None
    for t in LABELS:
        for p in LABELS:
            if t != p and res["confusion"][t][p] > best:
                best = res["confusion"][t][p]
                best_pair = (t, p)
    return {"pair": best_pair, "count": best}

check("S2", lambda: most_confused_pair(score_classification(GOLD_CLS, REF_ZERO_PRED, LABELS)))

[x] S2: PASS


### Stretch 3. A parse-and-repair loop

Some responses are malformed. Write `parse_with_repair(raw)` that tries
`parse_records`, and on failure attempts one repair: pull the first balanced
`{...}` block and parse that. Return the records and how many repairs were
needed, so you can report a repair rate.

In [18]:
def parse_with_repair(raw):
    try:
        return {"records": parse_records(raw), "repaired": 0}
    except ValueError:
        pass
    depth, start = 0, None
    for i, ch in enumerate(raw):
        if ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                obj = json.loads(raw[start:i + 1])
                return {"records": obj.get("records", []), "repaired": 1}
    raise ValueError("unrepairable")

_items = ",".join('{"id":"E%02d","label":"other"}' % i for i in range(1, 11))
_broken = ('oops the model rambled {"records": [' + _items
           + '], "stats": {"count": 10}} trailing junk } ]')

check("S3", lambda: {"records": len(parse_with_repair(_broken)["records"]),
                     "repaired": parse_with_repair(_broken)["repaired"]})

[x] S3: PASS


---
## Score yourself

Run `summary()` for your core count out of ten. Target at least eight of ten. If
you cleared the core tasks, try the stretch goals above.

In [19]:
summary()


CORE: 10/10 passing
STRETCH passing: S1, S2, S3
All core tasks green.


## Takeaways

- An eval harness is unit testing for model behavior. Build it before you argue
  about prompts.
- Few-shot examples reduce ambiguity, and the order you present them in shifts
  which boundary the model attends to.
- Accuracy hides per-class behavior. Read the confusion matrix.
- Chain-of-thought lifts reasoning accuracy, self-consistency stabilizes it, and
  both cost tokens you should measure.
- In production, prefer a private reason-then-answer contract and guard against
  leakage.

One honest caveat on the private variant. Asking a standard model to reason
silently and emit only the answer does not always keep the full reasoning
benefit, because the reasoning tokens are where the work happens. When you need
guaranteed reasoning without exposing it, use a model reasoning or thinking mode,
where those tokens are produced and billed separately. The leakage guard here is
about the output contract, not about hiding cost.